In [1]:
!pip install sentence-transformers
!pip install huggingface_hub --upgrade
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, models, util
import gc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.9/275.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.0/468.0 kB 20.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.26.2
    Uninstalling huggingface-hub-0.26.2:
      Successfully uninstalled huggingface-hub-0.26.2


In [2]:
def get_embeddings(words, model):
    # Compute embeddings without keeping unnecessary tensors in memory
    with torch.no_grad():
        return model.encode(words, convert_to_tensor=True, device=model.device, show_progress_bar=False)

def bootstrap_cosine_similarity(model, evidence_words, intuition_words, corpus_words, iterations=1000, save_every=100):
    """Performs bootstrap sampling, calculates cosine similarity, and saves raw values efficiently."""
    
    corpus_embeddings = get_embeddings(corpus_words, model)

    # Preallocate arrays
    evidence_scores_array = np.zeros((len(corpus_words), iterations), dtype=np.float32)
    intuition_scores_array = np.zeros((len(corpus_words), iterations), dtype=np.float32)

    # Perform bootstrap sampling
    for iteration in tqdm(range(1, iterations + 1), desc="Bootstrap iterations"):
        evidence_sample_size = np.random.randint(30, len(evidence_words) + 1)
        intuition_sample_size = np.random.randint(30, len(intuition_words) + 1)

        evidence_sample = np.random.choice(evidence_words, size=evidence_sample_size, replace=False)
        intuition_sample = np.random.choice(intuition_words, size=intuition_sample_size, replace=False)

        with torch.no_grad():
            evidence_sample_embeddings = torch.mean(get_embeddings(evidence_sample, model), dim=0)
            intuition_sample_embeddings = torch.mean(get_embeddings(intuition_sample, model), dim=0)

            evidence_similarity = util.cos_sim(corpus_embeddings, evidence_sample_embeddings).cpu().numpy().squeeze()
            intuition_similarity = util.cos_sim(corpus_embeddings, intuition_sample_embeddings).cpu().numpy().squeeze()

        # Store results in preallocated arrays
        evidence_scores_array[:, iteration - 1] = evidence_similarity
        intuition_scores_array[:, iteration - 1] = intuition_similarity

        # Free memory
        del evidence_sample_embeddings, intuition_sample_embeddings
        torch.cuda.empty_cache()
        gc.collect()

        # Save raw data in **efficient chunks**
        #if iteration % save_every == 0:
        #    raw_data_chunk = pd.DataFrame({
        #        "word": corpus_words,
         #       **{f"evidence_iter_{i+1}": evidence_scores_array[:, i] for i in range(iteration - save_every, iteration)},
         #       **{f"intuition_iter_{i+1}": intuition_scores_array[:, i] for i in range(iteration - save_every, iteration)}
         #   })
         #   raw_data_chunk.to_csv(f"bootstrap_results_raw_part_{iteration}.csv", index=False)

    # Compute summary statistics
    results_df = pd.DataFrame({"word": corpus_words})
    results_df["evidence_variance"] = np.var(evidence_scores_array, axis=1)
    results_df["evidence_mean"] = np.mean(evidence_scores_array, axis=1)
    results_df["evidence_median"] = np.median(evidence_scores_array, axis=1)
    results_df["evidence_min"] = np.min(evidence_scores_array, axis=1)
    results_df["evidence_max"] = np.max(evidence_scores_array, axis=1)
    results_df["evidence_25th_percentile"] = np.percentile(evidence_scores_array, 25, axis=1)
    results_df["evidence_75th_percentile"] = np.percentile(evidence_scores_array, 75, axis=1)
    
    results_df["intuition_variance"] = np.var(intuition_scores_array, axis=1)
    results_df["intuition_mean"] = np.mean(intuition_scores_array, axis=1)
    results_df["intuition_median"] = np.median(intuition_scores_array, axis=1)
    results_df["intuition_min"] = np.min(intuition_scores_array, axis=1)
    results_df["intuition_max"] = np.max(intuition_scores_array, axis=1)
    results_df["intuition_25th_percentile"] = np.percentile(intuition_scores_array, 25, axis=1)
    results_df["intuition_75th_percentile"] = np.percentile(intuition_scores_array, 75, axis=1)

    # Create final raw values DataFrame using **pd.concat()** (instead of adding columns one by one)
    #raw_values_list = [pd.DataFrame({
    #    f"evidence_iter_{i+1}": evidence_scores_array[:, i],
    #    f"intuition_iter_{i+1}": intuition_scores_array[:, i]
    #}) for i in range(iterations)]

    #results_raw_df = pd.concat([pd.DataFrame({"word": corpus_words})] + raw_values_list, axis=1)

    return results_df#, results_raw_df

In [3]:
evidence_words = pd.read_csv("/kaggle/input/dictionary2/PRODEMINFO_German_keywords.csv")['evidence'].dropna().tolist()
intuition_words = pd.read_csv("/kaggle/input/dictionary2/PRODEMINFO_German_keywords.csv")['intuition'].dropna().tolist()


In [4]:
corpus_df = pd.read_csv("/kaggle/input/final-data/youtube_corpus.csv")
corpus_words = pd.read_csv("/kaggle/input/final-data/youtube_corpus.csv")['word'].dropna().astype(str).tolist()
model = SentenceTransformer("/kaggle/input/sbert-model-yt/yt_model")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

results_df = bootstrap_cosine_similarity(
    model=model,
    evidence_words=evidence_words,
    intuition_words=intuition_words,
    corpus_words=corpus_words,
    iterations=1000
)

results_df["frequency"] = corpus_df["frequency"]

# Save the results
results_df.to_csv("youtube_corpus_bootstrap_statistic.csv", index=False)
#results_raw_df.to_csv("youtube_corpus_bootstrap_all.csv", index=False)

# Cleanup
del results_df, corpus_df, corpus_words, model
torch.cuda.empty_cache()
gc.collect()

Bootstrap iterations: 100%|██████████| 1000/1000 [07:20<00:00,  2.27it/s]


90

In [5]:
corpus_df = pd.read_csv("/kaggle/input/final-data/twitter_corpus.csv")
corpus_words = pd.read_csv("/kaggle/input/final-data/twitter_corpus.csv")['word'].dropna().astype(str).tolist()

model = SentenceTransformer("/kaggle/input/sbert-twitter/twitter_model")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

results_df  = bootstrap_cosine_similarity(
    model=model,
    evidence_words=evidence_words,
    intuition_words=intuition_words,
    corpus_words=corpus_words,
    iterations=1000
)

print("before frequency")
results_df["frequency"] = corpus_df["frequency"]

print("Save")
# Save the results
results_df.to_csv("twitter_corpus_bootstrap_statistic.csv", index=False)
#results_raw_df.to_csv("twitter_corpus_bootstrap_all.csv", index=False)


# Cleanup
del results_df, corpus_df, corpus_words, model
torch.cuda.empty_cache()
gc.collect()

Bootstrap iterations: 100%|██████████| 1000/1000 [09:00<00:00,  1.85it/s]


before frequency
Save


90

In [6]:
print("New load")
corpus_df = pd.read_csv("/kaggle/input/final-data/speeches_corpus.csv")
corpus_words = pd.read_csv("/kaggle/input/final-data/speeches_corpus.csv")['word'].dropna().astype(str).tolist()

print("New Model")
model = SentenceTransformer("/kaggle/input/sbert-model-new/model")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


print("New Bootstrapp")
results_df = bootstrap_cosine_similarity(
    model=model,
    evidence_words=evidence_words,
    intuition_words=intuition_words,
    corpus_words=corpus_words,
    iterations=1000
)

print("Frequency 2")
results_df["frequency"] = corpus_df["frequency"]


print("Final Save")
# Save the results
results_df.to_csv("speeches_corpus_bootstrap_statistic.csv", index=False)
#results_raw_df.to_csv("speeches_corpus_bootstrap_all.csv", index=False)

# Cleanup
del results_df, corpus_df, corpus_words, model
torch.cuda.empty_cache()
gc.collect()

New load
New Model
New Bootstrapp


Bootstrap iterations: 100%|██████████| 1000/1000 [06:57<00:00,  2.39it/s]


Frequency 2
Final Save


90